# UD2/UD3 - Regresión: House Prices

Cuaderno guía para EDA, preparación y modelado en el reto **House Prices** (Kaggle). Sigue los pasos de los documentos de EDA y preparación. Contiene celdas TODO comentadas para que el alumnado tome decisiones.

## Dependencias y configuración
- Ajusta rutas si los CSV están en otra carpeta.
- Descarga previa: `kaggle competitions download -c house-prices-advanced-regression-techniques`.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

sns.set_theme(style="whitegrid")
DATA_DIR = Path("data")
HOUSE_DIR = DATA_DIR / "house_prices"

## Utilidades

In [ ]:
def missing_report(df, top=20):
    miss = df.isna().mean().sort_values(ascending=False)
    return miss[miss > 0].head(top)

def eval_regression(y_true, y_pred, label="modelo"):
    metrics = {
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }
    print(label, metrics)
    return metrics

## Carga de datos

In [ ]:
train_hp_path = HOUSE_DIR / "train.csv"
test_hp_path = HOUSE_DIR / "test.csv"

if not train_hp_path.exists():
    raise FileNotFoundError(f"No se encontró {train_hp_path}. Descarga con Kaggle API y descomprime.")

hp_train = pd.read_csv(train_hp_path)
hp_test = pd.read_csv(test_hp_path) if test_hp_path.exists() else None
hp_train.shape, hp_test.shape if hp_test is not None else None

## EDA inicial

In [ ]:
hp_train.head()

In [ ]:
hp_train.info()

In [ ]:
hp_train.describe().T.head()

### Faltantes y tipos

In [ ]:
hp_missing = missing_report(hp_train, top=30); hp_missing

In [ ]:
num_cols_hp = hp_train.select_dtypes(include=[np.number]).columns.tolist(); cat_cols_hp = [c for c in hp_train.columns if c not in num_cols_hp]; (num_cols_hp[:10], cat_cols_hp[:10])

### Visualizaciones rápidas (opcionales)

In [ ]:
# Histograma de la variable objetivo
sns.histplot(hp_train['SalePrice'], kde=True)
plt.title('Distribución de SalePrice')
plt.tight_layout()

In [ ]:
# TODO: Añade más visuales (boxplots, correlaciones, pares clave)
# sns.boxplot(data=hp_train, x='OverallQual', y='SalePrice')

## Manejo de faltantes
- Ejemplo: eliminar columnas con >40% de NaNs y luego imputar numéricas con mediana, categóricas con moda.
- Deja celdas TODO para reglas específicas (por vecindario, tipo de garaje, etc.).

In [ ]:
missing_threshold = 0.4
cols_drop_hp = hp_missing[hp_missing > missing_threshold].index.tolist()

hp_train_clean = hp_train.drop(columns=cols_drop_hp)
if hp_test is not None:
    hp_test_clean = hp_test.drop(columns=cols_drop_hp)

num_cols_hp = hp_train_clean.select_dtypes(include=[np.number]).columns.tolist()
num_cols_hp.remove('SalePrice')
cat_cols_hp = [c for c in hp_train_clean.columns if c not in num_cols_hp + ['SalePrice']]
cols_drop_hp

In [ ]:
# TODO: Imputación específica por grupo (ej.: LotFrontage por vecindario)
# hp_train_clean['LotFrontage'] = hp_train_clean.groupby('Neighborhood')['LotFrontage'].transform(lambda s: s.fillna(s.median()))

## Split y pipelines de preparación

In [ ]:
X_hp = hp_train_clean.drop(columns=['SalePrice'])
y_hp = hp_train_clean['SalePrice']

X_train_hp, X_val_hp, y_train_hp, y_val_hp = train_test_split(
    X_hp, y_hp, test_size=0.2, random_state=42
)

numeric_pipe_hp = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe_hp = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocess_hp = ColumnTransformer([
    ("num", numeric_pipe_hp, num_cols_hp),
    ("cat", categorical_pipe_hp, cat_cols_hp),
])

## Modelos base y evaluación

In [ ]:
models_hp = {
    "LinearRegression": LinearRegression(),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

results_hp = {}
for name, model in models_hp.items():
    pipe = Pipeline([("prep", preprocess_hp), ("model", model)])
    pipe.fit(X_train_hp, y_train_hp)
    pred = pipe.predict(X_val_hp)
    results_hp[name] = eval_regression(y_val_hp, pred, label=name)
results_hp

## Optimización de hiperparámetros (ejemplo RandomForest)

In [ ]:
rf_hp = RandomForestRegressor(random_state=42, n_estimators=200)
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 4],
}

rf_pipe = Pipeline([("prep", preprocess_hp), ("model", rf_hp)])

grid_hp = GridSearchCV(rf_pipe, param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
# grid_hp.fit(X_hp, y_hp)  # Descomenta para entrenar (coste alto)
# grid_hp.best_params_, -grid_hp.best_score_

## Exportar limpio (opcional)

In [ ]:
# hp_train_clean.to_csv(DATA_DIR / 'house_prices_train_clean.csv', index=False)

## Notas finales
- Flujo: EDA → limpieza → split → pipeline → modelos → tuning.
- Activa las celdas TODO según tus decisiones.